# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset identifier: {metadata.identifier}")
print(f"Published on: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by their `@id` as required by Croissant. This helps understand how the dataset is structured and what information each part contains.

In [ ]:
# List RecordSet, Field, and Column @id's available in the dataset
record_sets = list(metadata.recordSet)
print(f"Total record sets in this dataset: {len(record_sets)}\n")

for rs in record_sets:
    print(f"=== RecordSet @id: {rs['@id']} ===")
    # List associated fields for each record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields and their @ids:")
    for f in fields:
        print(f"  - {f['@id']}  ({f.get('name', 'no name')})")
        # If the field has columns (for further granularity)
        columns = f.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("    Columns:")
            for c in columns:
                print(f"      * {c['@id']}  ({c.get('name', 'no name')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

> **Note:** The record set and field `@id`s below are placeholders. Edit them to match real values in the dataset structure found above.

In [ ]:
# Replace with actual RecordSet @id found above; here we look for the main record set used for protein data
# (If not known, print record_sets variable from above and pick an @id)

main_record_set_id = None
for rs in record_sets:
    # Try to choose the record set most likely containing the protein table (usually the largest one in proteomics)
    rs_name = rs.get('name', '').lower()
    if 'protein' in rs_name or 'table' in rs_name:
        main_record_set_id = rs['@id']
        break
    if not main_record_set_id:
        main_record_set_id = rs['@id']
if not main_record_set_id:
    raise ValueError("Couldn't find suitable record set.")

# Example: main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/records/12345'

# Load all record sets into DataFrames, referencing each by @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    # Use mlcroissant's records generator
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs_id}")

print(f"Columns of main DataFrame ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data, or grouping data by attributes.

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with actual `@id`s or column names as printed above. For demonstration, common proteomics fields such as 'MW' (molecular weight) or 'abundance' may be used if present.

In [ ]:
# Choose a numeric field (e.g., protein molecular weight or abundance)
# For demo, we try to find a suitable numeric column
sample_df = dataframes[main_record_set_id]
numeric_column_candidates = [c for c in sample_df.columns if sample_df[c].dtype.kind in 'iuf']
if numeric_column_candidates:
    numeric_field = numeric_column_candidates[0]
else:
    # Fallback: try to parse float columns
    for c in sample_df.columns:
        try:
            sample_df[c] = pd.to_numeric(sample_df[c])
            numeric_field = c
            break
        except Exception:
            continue
    else:
        raise ValueError("No numeric field found for EDA.")

print(f"Using numeric field: {numeric_field}")

# Filter records for values above a threshold
threshold = sample_df[numeric_field].mean() if sample_df[numeric_field].dtype.kind in 'iuf' else 10
filtered_df = sample_df[sample_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the chosen numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field/category
possible_group_columns = [c for c in sample_df.columns if c != numeric_field]
group_field = None
# Pick the first string/categorical column
for c in possible_group_columns:
    if sample_df[c].dtype == object and not sample_df[c].str.contains('^[0-9\.]+$').all():
        group_field = c
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is a histogram and scatterplot for the selected numeric field and a categorical attribute (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(sample_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If we have a group_field, let's see boxplots
if group_field:
    plt.figure(figsize=(12,5))
    # Only top 10 categories
    top_groups = sample_df[group_field].value_counts().nlargest(10).index
    sns.boxplot(
        data=sample_df[sample_df[group_field].isin(top_groups)],
        x=group_field,
        y=numeric_field
    )
    plt.title(f"{numeric_field} by {group_field} (top 10)")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
In this notebook, we have loaded, inspected, and visualized the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. All data entities are referenced by their `@id` to ensure consistency. Key findings include basic distributions of protein attributes, and this workflow can be extended for further detailed proteomic or statistical analysis.

For in-depth work, consult the metadata and Croissant file for exact field/categorical meanings, and reference domain-specific publications in the dataset's citation field.